Name: Rujuta Bhanose
PRN: 22311432
Roll No: 381042
Batch: P1

In [16]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# Preprocesing

In [17]:
df = pd.read_csv("student_performance_dataset.csv")
print("Original shape:", df.shape)
df.head()

Original shape: (100, 4)


,study_hours,attendance,internal_marks,final_score
0,3.31,41.89,29.26,34.23
1,7.63,78.18,12.52,44.91
2,5.99,58.86,14.85,50.04
3,4.99,70.51,36.96,53.43
4,1.67,94.45,28.19,35.07


In [18]:
print("Missing values in df:")
print(df.isna().sum())

Missing values in df:
study_hours       2
attendance        1
internal_marks    4
final_score       0
dtype: int64


In [19]:
df_clean = df.copy()

numeric_cols = df_clean.select_dtypes(include=[np.number]).columns

for col in numeric_cols:
    mean_value = df_clean[col].mean()
    df_clean[col] = df_clean[col].fillna(mean_value)

print("Missing values after imputation:")
print(df_clean.isna().sum())

Missing values after imputation:
study_hours       0
attendance        0
internal_marks    0
final_score       0
dtype: int64


In [20]:
scaler = MinMaxScaler(feature_range=(0, 1))

df_norm = df_clean.copy()
df_norm[numeric_cols] = scaler.fit_transform(df_clean[numeric_cols])

print("First 5 rows after normalization:")
df_norm.head()

First 5 rows after normalization:


,study_hours,attendance,internal_marks,final_score
0,0.376359,0.025034,0.646701,0.351591
1,0.963315,0.643052,0.080203,0.582810
2,0.740489,0.314033,0.159052,0.693873
3,0.604620,0.512432,0.907276,0.767266
4,0.153533,0.920129,0.610491,0.369777


In [21]:
df_clean.to_csv("student_performance_clean.csv", index=False)
df_norm.to_csv("student_performance_normalized.csv", index=False)

print("Saved:")
print("- student_performance_clean.csv")
print("- student_performance_normalized.csv")

Saved:
- student_performance_clean.csv
- student_performance_normalized.csv


# Models

In [22]:
import os

NUM_CLIENTS = 4

CLIENT_DATA_DIR = "clients_data"
os.makedirs(CLIENT_DATA_DIR, exist_ok=True)

# Use the imputed and normalized data instead of original df
df_shuffled = df_norm.sample(frac=1.0, random_state=42).reset_index(drop=True)

# Split the DataFrame into chunks
chunk_size = len(df_shuffled) // NUM_CLIENTS
client_datasets = [df_shuffled.iloc[i*chunk_size:(i+1)*chunk_size] for i in range(NUM_CLIENTS)]

client_files = []

for client_id, client_df in enumerate(client_datasets, start=1):
    client_filename = os.path.join(CLIENT_DATA_DIR, f"client_{client_id}.csv")
    client_df.to_csv(client_filename, index=False)
    client_files.append(client_filename)
    print(f"Client {client_id} dataset saved to {client_filename} with shape {client_df.shape}")

Client 1 dataset saved to clients_data/client_1.csv with shape (25, 4)
Client 2 dataset saved to clients_data/client_2.csv with shape (25, 4)
Client 3 dataset saved to clients_data/client_3.csv with shape (25, 4)
Client 4 dataset saved to clients_data/client_4.csv with shape (25, 4)


In [23]:
# Here we fit a separate regression model per client on its own private data.
# # Only model parameters are returned (not raw data), similar in spirit to federated learning. [web:28]

from sklearn.linear_model import LinearRegression

def train_local_model(client_data_path):
    data = pd.read_csv(client_data_path)
    
    X = data[["study_hours", "attendance", "internal_marks"]]
    y = data["final_score"]
    
    model = LinearRegression()
    model.fit(X, y)
    
    # Return learned parameters instead of raw data
    return {
        "coef_": model.coef_,
        "intercept_": model.intercept_,
        "n_samples": len(data)
    }

local_models = []
for client_path in client_files:
    params = train_local_model(client_path)
    local_models.append(params)
    print(f"Trained local model from {client_path}:")
    print("  Coefficients:", params["coef_"])
    print("  Intercept:", params["intercept_"])
    print("  Samples:", params["n_samples"])


Trained local model from clients_data/client_1.csv:
  Coefficients: [0.41241435 0.31250283 0.3539165 ]
  Intercept: -0.05898025209928859
  Samples: 25
Trained local model from clients_data/client_2.csv:
  Coefficients: [0.42901375 0.44503662 0.29778305]
  Intercept: -0.019066355972333304
  Samples: 25
Trained local model from clients_data/client_3.csv:
  Coefficients: [0.53254802 0.39979451 0.39325697]
  Intercept: -0.09014194159702205
  Samples: 25
Trained local model from clients_data/client_4.csv:
  Coefficients: [0.51325505 0.44144505 0.32958175]
  Intercept: -0.07024796047739079
  Samples: 25


In [24]:
# Simple weighted averaging of local models
import numpy as np

def aggregate_models(local_models):
    # Weighted by number of samples per client
    total_samples = sum(m["n_samples"] for m in local_models)
    
    # Stack coefficients
    coefs = np.array([m["coef_"] * m["n_samples"] for m in local_models])
    intercepts = np.array([m["intercept_"] * m["n_samples"] for m in local_models])
    
    # Weighted average
    global_coef = coefs.sum(axis=0) / total_samples
    global_intercept = intercepts.sum() / total_samples
    
    return {
        "coef_": global_coef,
        "intercept_": global_intercept
    }

global_model = aggregate_models(local_models)
print("Global model parameters (aggregated):")
print("  Coefficients:", global_model["coef_"])
print("  Intercept:", global_model["intercept_"])

Global model parameters (aggregated):
  Coefficients: [0.47180779 0.39969475 0.34363457]
  Intercept: -0.05960912753650868


In [25]:
import pickle

with open('global_model.pkl', 'wb') as f:
    pickle.dump(global_model, f)

print("Global model saved to global_model.pkl")

Global model saved to global_model.pkl
